In [1]:
# Importing packages
import pandas as pd
import torch
import os
import math
os.environ["HF_HUB_DISABLE_PROGREvSS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F
from beir.retrieval.evaluation import EvaluateRetrieval

/work/mbouthil/.conda/envs/myuwenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Loading Data ###
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, dev_queries, dev_qrels = GenericDataLoader(data_folder=data_dir).load(split="dev")
dev_info = [(key, value) for key, value in dev_queries.items()]

100%|██████████| 8841823/8841823 [01:16<00:00, 115846.31it/s]


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    f"/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/query_encoder_syn_que_final"
).to(device)
query_encoder.eval()

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 196.95it/s, Materializing param=pooler.dense.weight]                               


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [8]:
def encode_query(query:str) -> Tensor:

    queries = [query] if isinstance(query, str) else query
    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            queries, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ).to(device)

    # Mean Pooling
    out = query_encoder(**inputs)
    last_hidden_state = out.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).float()
    emb = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

    emb = F.normalize(emb, p=2, dim=-1)
    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

In [9]:
q_emb_1 = encode_query("What is the capital of France?").detach().cpu().numpy()
q_emb_2 = encode_query("How to bake a cake ").detach().cpu().numpy()

score = np.dot(q_emb_1[0], q_emb_2[0])
print(score)

-0.0103813885


In [12]:
index = faiss.read_index(f"/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_syn_que_final.index")
print(index.ntotal)

8841823


In [ ]:
index = faiss.read_index(f"/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_syn_que_final.index")
print(index.ntotal)

# pick a dev query
qid = list(dev_qrels.keys())[0]
query_text = dev_queries[qid]

# encode
q_emb = encode_query(query_text).detach().cpu().numpy()

# search
scores, pids = index.search(q_emb, 100)

gold = set(dev_qrels[qid].keys())
retrieved = set(str(pid) for pid in pids[0])

In [36]:
# print("Gold:", list(gold)[:5])
# print("Retrieved:", list(retrieved)[:5])
# print("Intersection:", gold & retrieved)
# print("\n")
# print(dev_queries[qid])
# print("\n")
for id in list(retrieved)[:5]:
    print(corpus[id]['text'])

Peters Colony was comprised of 23 North Texas counties, or parts thereof, in the fourth and final colonization contract signed in 1843.Actual settlement of Peters Colony had begun in 1841 with a contact to '...eters Colony was comprised of 23 North Texas counties, or parts thereof, in the fourth and final colonization contract signed in 1843.
Named in honor of James Mendenhall and founded in 1816, Jamestown, NC was the earliest continuing settlement in the Piedmont region. More recently, archeologists have found evidence of nearby human habitation dating back thousands of years.
The Mayflower Compact We whose names are underwritten, the loyal subjects of our dread Sovereign Lord King James, by the Grace of God of Great Britain, France and Ireland, King, Defender of the Faith, etc.
Peters Colony (Peters' Colony) is a name applied to four empresario land grant contracts first by the Republic of Texas and then the State of Texas for settlement in north Texas. The contracts were signed gro

In [29]:
passage_encoder = AutoModel.from_pretrained(
    f"/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/passage_encoder_syn_que_final"
).to(device)
passage_encoder.eval()

def encode_passage(passage:str) -> Tensor:

    queries = [passage] if isinstance(passage, str) else passage
    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            queries, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=128
        ).to(device)

    # Mean Pooling
    out = passage_encoder(**inputs)
    last_hidden_state = out.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).float()
    emb = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

    emb = F.normalize(emb, p=2, dim=-1)
    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 227.76it/s, Materializing param=pooler.dense.weight]                               


In [30]:
gold_pid = list(gold)[0]
gold_text = corpus[gold_pid]["text"]

gold_emb = encode_passage([gold_text]).detach().cpu().numpy()

print("Similarity(query, gold):",
      np.dot(q_emb[0], gold_emb[0]))

Similarity(query, gold): 0.99840623


In [31]:
for key in corpus.keys():
    print(corpus[key]['text'])
    break

The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.


In [32]:
gold_pid = 7067032
faiss_vec = index.reconstruct(int(gold_pid))

In [33]:
print("Cosine(indexed, fresh):",
      np.dot(faiss_vec, gold_emb[0]))

Cosine(indexed, fresh): 1.0


In [34]:
scores, pids = index.search(q_emb, 100)

print("Top score:", scores[0][0])
print("Gold score manual:",
      np.dot(q_emb[0], gold_emb[0]))

Top score: 0.9995092
Gold score manual: 0.99840623


In [35]:
int(gold_pid) in pids[0]

False